Atividade 1 — Item a)

1. Resposta Analítica e Teórica

A condição de complementaridade de Karush-Kuhn-Tucker (KKT) exige formalmente que o produto clássico e lógico entre o multiplicador de Lagrange $\mu_j$ e a função de restrição de desigualdade $g_j(x)$ seja rigorosamente nulo no ponto ótimo, isto é:$$\mu_j \cdot g_j(x^*) = 0$$Quando o código detecta em tempo de execução que uma determinada restrição está estritamente inativa (como no exemplo dado, onde $g_1(x_k) = -5.4 < 0$), significa geometricamente que o ponto corrente se localiza no interior do espaço viável em relação a essa fronteira específica. Uma vez que $g_1(x_k) \neq 0$, a única forma de satisfazer a igualdade de complementaridade sem violar o sistema matemático é garantir que o multiplicador associado anule a expressão.Portanto, o valor exato que deve ser atribuído na memória do computador para o multiplicador é $\mu_1 = 0.0$ (zero absoluto em ponto flutuante).

2. Implementação Computacional em Python

In [ ]:
import numpy as np

def resolver_item_1a(g1_val, mu1_val):
    """
    Avalia uma restrição de desigualdade g1(x) e atribui o valor exato
    ao multiplicador mu1 na memória caso ela esteja estritamente inativa.
    """
    # Convertendo para float padrão
    g1 = float(g1_val)
    mu1 = float(mu1_val)

    # Tolerância estrita para segurança numérica
    epsilon = 1e-8

    # Verificação lógica: Se g1(x) < -epsilon, a restrição está estritamente inativa
    if g1 < -epsilon:
        mu1_corrigido = 0.0
        status = "Inativa (Forçado mu1 = 0.0)"
    else:
        mu1_corrigido = mu1
        status = "Ativa ou na fronteira de tolerância"

    return mu1_corrigido, status

# Teste prático com os dados do enunciado
if __name__ == "__main__":
    g1_x = -5.4
    mu1_corrente = 2.75  # Valor genérico em uma iteração intermediária

    mu1_final, situacao = resolver_item_1a(g1_x, mu1_corrente)

    print(f"Valor de g1(x): {g1_x}")
    print(f"Multiplicador original na memória: {mu1_corrente}")
    print(f"Estado da restrição: {situacao}")
    print(f"Valor exato de mu1 atribuído na memória: {mu1_final}")

Valor de g1(x): -5.4
Multiplicador original na memória: 2.75
Estado da restrição: Inativa (Forçado mu1 = 0.0)
Valor exato de mu1 atribuído na memória: 0.0


Item b)

1. Resposta Analítica e Teórica

Em computação científica e álgebra linear computacional, testar a igualdade exata através de operadores lógicos rígidos (como mu_j * g_j(x) == 0.0) em variáveis de precisão contínua (float ou double) é extremamente perigoso e desaconselhável. Isso ocorre devido à representação binária finita dos números reais sob o padrão IEEE 754. Durante as iterações de um algoritmo de otimização, operações sucessivas de ponto flutuante acumulam inevitavelmente pequenos erros de arredondamento, truncamento e cancelamento catastrófico. Como consequência, uma folga ou multiplicador que analiticamente deveria ser nulo, na memória da máquina assume um valor residual microscópico (por exemplo, 1.42e-16). O uso de uma comparação exata faz com que o programa rejeite pontos ótimos perfeitamente válidos, aprisionando o solver em loops infinitos ou causando falsos alarmes de não convergência.Para contornar essa limitação e garantir a robustez do código, formula-se o resíduo de complementaridade ($Res_{comp}$) utilizando uma tolerância estrita $\epsilon = 10^{-8}$. Em vez de buscar o zero absoluto, o critério numérico aceita o ponto se o valor absoluto do produto escalar estiver confinado dentro desta vizinhança controlada. A formulação matemática robusta para um conjunto de restrições de desigualdade é expressa por:$$Res_{comp} = \max_{j} \left| \mu_j \cdot g_j(x_k) \right| \leq \epsilon, \quad \text{com } \epsilon = 10^{-8}$$

2. Implementação Computacional em Python

In [ ]:
import numpy as np

def verificar_complementaridade_robusta(g_vetor, mu_vetor, epsilon=1e-8):
    """
    Implementa o teste de complementaridade KKT de forma robusta para computação científica,
    mitigando os riscos de erros de ponto flutuante através de uma tolerância epsilon.
    """
    # Garante a conversão para arranjos numéricos do NumPy do tipo float
    g = np.atleast_1d(np.array(g_vetor, dtype=float))
    mu = np.atleast_1d(np.array(mu_vetor, dtype=float))

    # Calcula os resíduos individuais de complementaridade por componente usando valor absoluto
    residuos_individuais = np.abs(mu * g)

    # Determina o resíduo máximo do sistema (Norma do Supremo / Norma Infinita)
    residuo_max_kkt = np.max(residuos_individuais)

    # Avaliação do critério numérico robusto ao invés de usar '== 0.0'
    condicao_satisfeita = residuo_max_kkt <= epsilon

    return residuo_max_kkt, condicao_satisfeita

# Teste prático simulando ruído numérico de ponto flutuante
if __name__ == "__main__":
    # Cenário: g2(x) teoricamente é 0.0 (restrição ativa), mas possui ruído de precisão infinitesimal
    g_teste = np.array([-1.25e-16, -2.40])
    mu_teste = np.array([3.50, 0.0])       # mu1 ativo, mu2 inativo

    epsilon_alvo = 1e-8
    residuo, status = verificar_complementaridade_robusta(g_teste, mu_teste, epsilon=epsilon_alvo)

    print(f"Vetor g(x) com ruído: {g_teste}")
    print(f"Vetor mu: {mu_teste}")
    print(f"Resíduo de complementaridade calculado: {residuo:.4e}")
    print(f"O resíduo limpa a memória sob a tolerância {epsilon_alvo}? {status}")

Vetor g(x) com ruído: [-1.25e-16 -2.40e+00]
Vetor mu: [3.5 0. ]
Resíduo de complementaridade calculado: 4.3750e-16
O resíduo limpa a memória sob a tolerância 1e-08? True


Atividade 2 — Item a)

1. Resposta Analítica e TeóricaO enunciado da Atividade 2 propõe avaliar as condições de regularidade (Qualificação de Restrições por Independência Linear — LICQ) para um sistema com duas restrições de igualdade:$$h_1(x) = x_1 + x_2 - 2 = 0$$$$h_2(x) = x_1^2 + x_2^2 - 2 = 0$$Para montar a matriz Jacobiana $J(x)$ das restrições de igualdade, devemos calcular os gradientes transpostos de cada restrição em relação às variáveis primais $x_1$ e $x_2$. A estrutura geral da matriz Jacobiana $J(x) \in \mathbb{R}^{m \times n}$ para $m=2$ restrições e $n=2$ variáveis é dada por:$$J(x) = \begin{bmatrix} \nabla h_1(x)^T \\ \nabla h_2(x)^T \end{bmatrix} = \begin{bmatrix} \frac{\partial h_1}{\partial x_1} & \frac{\partial h_1}{\partial x_2} \\ \frac{\partial h_2}{\partial x_1} & \frac{\partial h_2}{\partial x_2} \end{bmatrix}$$Calculando as derivadas parciais:

Para $h_1(x)$:$$\frac{\partial h_1}{\partial x_1} = 1, \quad \frac{\partial h_1}{\partial x_2} = 1 \implies \nabla h_1(x)^T = \begin{bmatrix} 1 & 1 \end{bmatrix}$$Para $h_2(x)$:$$\frac{\partial h_2}{\partial x_1} = 2x_1, \quad \frac{\partial h_2}{\partial x_2} = 2x_2 \implies \nabla h_2(x)^T = \begin{bmatrix} 2x_1 & 2x_2 \end{bmatrix}$$Portanto, a matriz Jacobiana analítica geral é:$$J(x) = \begin{bmatrix} 1 & 1 \\ 2x_1 & 2x_2 \end{bmatrix}$$Avaliando especificamente no ponto de teste numérico sugerido $x_k = [1, 1]^T$ (ou seja, $x_1 = 1$ e $x_2 = 1$), substituímos os valores na matriz:$$J(1, 1) = \begin{bmatrix} 1 & 1 \\ 2(1) & 2(1) \end{bmatrix} = \begin{bmatrix} 1 & 1 \\ 2 & 2 \end{bmatrix}$$

2. Implementação Computacional em Python

In [ ]:
import numpy as np

def calcular_jacobiana_item_2a(x_ponto):
    """
    Calcula analiticamente e avalia numericamente a matriz Jacobiana
    das restrições h1(x) e h2(x) no ponto x_ponto fornecido.
    """
    x1 = float(x_ponto[0])
    x2 = float(x_ponto[1])

    # Construção da matriz Jacobiana J(x) avaliada no ponto
    J = np.array([
        [1.0, 1.0],        # Derivadas parciais de h1 em relação a x1 e x2
        [2.0*x1, 2.0*x2]   # Derivadas parciais de h2 em relação a x1 e x2
    ], dtype=float)

    return J

# Teste prático com o ponto do enunciado
if __name__ == "__main__":
    ponto_teste = np.array([1.0, 1.0])

    jacobiana_avaliada = calcular_jacobiana_item_2a(ponto_teste)

    print("=== RESOLUÇÃO DA ATIVIDADE 2 - ITEM A ===")
    print(f"Ponto de teste x_k: {ponto_teste}")
    print("Matriz Jacobiana J(1,1) computada:")
    print(jacobiana_avaliada)

=== RESOLUÇÃO DA ATIVIDADE 2 - ITEM A ===
Ponto de teste x_k: [1. 1.]
Matriz Jacobiana J(1,1) computada:
[[1. 1.]
 [2. 2.]]


Item b)

1. Resposta Analítica e Teórica

Para avaliar por que um solver falharia catastroficamente ao tentar computar os multiplicadores de Lagrange usando uma abordagem baseada em $(J J^T)^{-1}$, calculamos inicialmente o determinante da matriz Jacobiana $J(1, 1)$ obtida no item anterior:$$J(1,1) = \begin{bmatrix} 1 & 1 \\ 2 & 2 \end{bmatrix}$$O determinante dessa matriz quadrada é:$$\det(J) = (1 \cdot 2) - (1 \cdot 2) = 2 - 2 = 0$$Como $\det(J) = 0$, a matriz Jacobiana é singular e tem posto incompleto ($\text{posto}(J) = 1 < 2$). Isso significa que as linhas de $J(1,1)$ são linearmente dependentes (a segunda linha é exatamente o dobro da primeira). Consequentemente, a Qualificação de Restrições por Independência Linear (LICQ) é violada no ponto $x_k = [1,1]^T$.Muitos métodos numéricos e equações de projeção estimam os multiplicadores de Lagrange $\lambda$ resolvendo o sistema de mínimos quadrados normais, cuja solução analítica envolve a matriz quadrada $J J^T \in \mathbb{R}^{2 \times 2}$:$$J J^T = \begin{bmatrix} 1 & 1 \\ 2 & 2 \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 1 & 2 \end{bmatrix} = \begin{bmatrix} 1+1 & 2+2 \\ 2+2 & 4+4 \end{bmatrix} = \begin{bmatrix} 2 & 4 \\ 4 & 8 \end{bmatrix}$$Se calcularmos o determinante de $J J^T$:$$\det(J J^T) = (2 \cdot 8) - (4 \cdot 4) = 16 - 16 = 0$$Como $J J^T$ também é singular, a sua inversa $(J J^T)^{-1}$ não existe. Ao tentar executar essa inversão, qualquer algoritmo computacional sofrerá uma divisão por zero (ou gerará valores nulos no determinante), resultando em um erro de estouro numérico (NaN ou Inf), colapsando o cálculo dos multiplicadores e a projeção de direções viáveis. Conforme a Seção 2.2 de Martínez e Santos, a falta de regularidade impede a garantia de que as forças (gradientes) das restrições gerem uma base única e estável para mapear o espaço dual.

2. Implementação Computacional em Python
Python

In [ ]:
import numpy as np

def avaliar_falha_licq_item_2b(J):
    """
    Avalia o posto, o determinante e tenta computar a matriz (J J^T)^-1
    para demonstrar numericamente a falha catastrófica por perda de regularidade.
    """
    # 1. Calcular o posto da matriz Jacobiana
    posto = np.linalg.matrix_rank(J)

    # 2. Computar o produto J * J^T
    JJ_T = np.dot(J, J.T)

    # 3. Calcular o determinante de J J^T
    det_JJ_T = np.linalg.det(JJ_T)

    print(f"Posto da Matriz J: {posto} (Esperado para LICQ: 2)")
    print(f"Determinante de (J J^T): {det_JJ_T:.4f}")

    # 4. Tentar inverter a matriz dentro de um bloco try-except para capturar a falha
    try:
        J_inversa = np.linalg.inv(JJ_T)
        status = "Sucesso na inversão (Inesperado)"
    except np.linalg.LinAlgError as e:
        J_inversa = None
        status = f"FALHA CATASTRÓFICA RECONHECIDA: {e}"

    return JJ_T, J_inversa, status

if __name__ == "__main__":
    # Matriz Jacobiana calculada no item A para o ponto (1,1)
    J_avaliada = np.array([[1.0, 1.0],
                           [2.0, 2.0]], dtype=float)

    print("=== RESOLUÇÃO DA ATIVIDADE 2 - ITEM B ===")
    matriz_bloco, inversa, resultado_status = avaliar_falha_licq_item_2b(J_avaliada)
    print(f"Resultado da tentativa de inversão: {resultado_status}")

=== RESOLUÇÃO DA ATIVIDADE 2 - ITEM B ===
Posto da Matriz J: 1 (Esperado para LICQ: 2)
Determinante de (J J^T): 0.0000
Resultado da tentativa de inversão: FALHA CATASTRÓFICA RECONHECIDA: Singular matrix


Atividade 3 — Formulação do Critério de Parada (KKT Error)

1. Resposta Analítica e Mapeamento Logístico

Para construir um resolvedor (solver) numérico robusto de otimização, precisamos unificar as 4 condições clássicas de Karush-Kuhn-Tucker (Estacionaridade, Viabilidade Primal de Igualdade, Viabilidade Primal de Desigualdade e Complementaridade) em um indicador numérico escalar conhecido como Erro KKT ($E_{\text{KKT}}$). Esse escalar funciona como o critério de parada do algoritmo.Para estruturar a lógica computacional, preenchemos o mapeamento matemático e vetorial de cada resíduo conforme solicitado no PDF:


O mapeamento lógico do vetor de resíduos monitorado pela unidade central de processamento é apresentado na tabela abaixo:

| Dimensão do Erro KKT | Condição Associada | Expressão Matemática Computacional | Dimensão Vetorial | Significado Prático no Processador |
| :--- | :--- | :--- | :--- | :--- |
| **Resíduo 1 ($R_{\text{est}}$)** | Estacionaridade | $\nabla f(x) + \sum_{i=1}^m \lambda_i \nabla h_i(x) + \sum_{j=1}^p \mu_j \nabla g_j(x)$ | $\mathbb{R}^n$ ($n$ variáveis) | Mede o desbalanço de forças entre o gradiente da objetivo e os das restrições. |
| **Resíduo 2 ($R_{\text{eq}}$)** | Viabilidade Primal ($=0$) | $h(x)$ | $\mathbb{R}^m$ ($m$ igualdades) | Afastamento absoluto das restrições de igualdade em relação a zero. |
| **Resíduo 3 ($R_{\text{ineq}}$)** | Viabilidade Primal ($\leq 0$) | $\max(0, g(x))$ | $\mathbb{R}^p$ ($p$ desigualdades) | Penaliza estritamente as violações onde $g_j(x) > 0$. Se $g_j(x) \leq 0$, o erro é zero. |
| **Resíduo 4 ($R_{\text{comp}}$)** | Complementaridade | $\mu \odot g(x)$ | $\mathbb{R}^p$ ($p$ desigualdades) | Produto elemento a elemento ($\odot$). Garante que ou a restrição está ativa ($g_j=0$) ou a força é nula ($\mu_j=0$). |

A unificação dos blocos operacionais em memória é dada pela aplicação da norma do supremo ($\|\cdot\|_\infty$). O *solver* encerra o processo iterativo no instante em que o critério consolidado satisfaz a tolerância:

$$E_{\text{KKT}} = \max \left( \|R_{\text{est}}\|_\infty, \|R_{\text{eq}}\|_\infty, \|R_{\text{ineq}}\|_\infty, \|R_{\text{comp}}\|_\infty \right) \leq \epsilon_{\text{alvo}}$$

2. Implementação Computacional em Python

In [ ]:
import numpy as np

def calcular_erro_kkt_consolidado(grad_f, J_h=None, h_val=None, J_g=None, g_val=None, lambda_val=None, mu_val=None):
    """
    ATIVIDADE 3
    Unifica e calcula as 4 dimensões de resíduos KKT de forma robusta e matricial.
    """
    # 0. Garante que todas as entradas sejam arrays do NumPy do tipo float64
    grad_f = np.array(grad_f, dtype=float).flatten()
    h = np.array(h_val, dtype=float).flatten() if h_val is not None else np.array([])
    g = np.array(g_val, dtype=float).flatten() if g_val is not None else np.array([])
    lam = np.array(lambda_val, dtype=float).flatten() if lambda_val is not None else np.array([])
    mu = np.array(mu_val, dtype=float).flatten() if mu_val is not None else np.array([])

    # 1. Resíduo de Estacionaridade (R_est) -> Dimensão: R^n
    R_est = np.copy(grad_f)
    if h.size > 0 and J_h is not None:
        R_est += np.dot(np.array(J_h, dtype=float).T, lam)
    if g.size > 0 and J_g is not None:
        R_est += np.dot(np.array(J_g, dtype=float).T, mu)

    # 2. Resíduo de Viabilidade Primal de Igualdade (R_eq) -> Dimensão: R^m
    R_eq = np.copy(h)

    # 3. Resíduo de Viabilidade Primal de Desigualdade (R_ineq) -> Dimensão: R^p
    # Formulação robusta: max(0, g(x)) penaliza apenas violações (onde g(x) > 0)
    R_ineq = np.maximum(0.0, g) if g.size > 0 else np.array([])

    # 4. Resíduo de Complementaridade (R_comp) -> Dimensão: R^p
    # Produto de Hadamard (elemento a elemento) entre multiplicadores e restrições
    R_comp = mu * g if g.size > 0 else np.array([])

    # 5. Extração das Normas Infinitas (Norma do Supremo: valor máximo absoluto)
    norm_est = np.max(np.abs(R_est)) if R_est.size > 0 else 0.0
    norm_eq = np.max(np.abs(R_eq)) if R_eq.size > 0 else 0.0
    norm_ineq = np.max(np.abs(R_ineq)) if R_ineq.size > 0 else 0.0
    norm_comp = np.max(np.abs(R_comp)) if R_comp.size > 0 else 0.0

    # 6. Erro KKT Máximo Consolidado (Critério Global de Parada)
    erro_kkt_max = max(norm_est, norm_eq, norm_ineq, norm_comp)

    return {
        "Erro_KKT_Consolidado": erro_kkt_max,
        "Residuo_Estacionaridade": norm_est,
        "Residuo_Igualdade": norm_eq,
        "Residuo_Desigualdade_Violada": norm_ineq,
        "Residuo_Complementaridade": norm_comp
    }

# --- Bloco de Teste Oficial ---
if __name__ == "__main__":
    # Testando com um cenário padrão para validar o comportamento da função
    gradiente_f = np.array([0.01, -0.02])
    jacobiana_g = np.array([[2.0, 1.0]])
    valor_g = np.array([-1.5e-9])
    multiplicador_mu = np.array([0.012])

    resultado = calcular_erro_kkt_consolidado(
        grad_f=gradiente_f,
        J_g=jacobiana_g,
        g_val=valor_g,
        mu_val=multiplicador_mu
    )

    print("======= VERIFICAÇÃO DO ERRO KKT =======")
    print(f"-> ERRO MAXIMO CONSOLIDADO: {resultado['Erro_KKT_Consolidado']:.6e}")
    print(f"   Norma Estacionaridade:  {resultado['Residuo_Estacionaridade']:.6e}")
    print(f"   Norma Igualdade:         {resultado['Residuo_Igualdade']:.6e}")
    print(f"   Norma Desigualdade:      {resultado['Residuo_Desigualdade_Violada']:.6e}")
    print(f"   Norma Complementaridade: {resultado['Residuo_Complementaridade']:.6e}")
    print("===========================================================")

======= VERIFICAÇÃO DO ERRO KKT =======
-> ERRO MAXIMO CONSOLIDADO: 3.400000e-02
   Norma Estacionaridade:  3.400000e-02
   Norma Igualdade:         0.000000e+00
   Norma Desigualdade:      0.000000e+00
   Norma Complementaridade: 1.800000e-11


Atividade 4 — Arquitetura de um Resolvedor (Validador de Pontos Críticos)

1. Resposta Analítica e Teórica

A Atividade 4 estipula a criação da lógica de controle estrutural de um resolvedor (solver) voltado a validar se um ponto arbitrário $x_k$ e seus respectivos multiplicadores duais ($\lambda_k, \mu_k$) atendem aos critérios de otimalidade e viabilidade matemática definidos pelo sistema KKT.

Diferente do cálculo isolado de resíduos (realizado na Atividade 3), a arquitetura de um validador inteligente exige a tomada de decisões em cascata e o tratamento lógico de tolerâncias para classificar o estado atual do sistema de otimização. O resolvedor baseia-se nos seguintes pilares teóricos:

Vetor de Estados Primal-Dual:

 O processador recebe o ponto $x_k \in \mathbb{R}^n$, as funções de restrições avaliadas $h(x_k)$ e $g(x_k)$, a matriz Jacobiana correspondente e o vetor de multiplicadores $\lambda_k \in \mathbb{R}^m$ (para igualdades) e $\mu_k \in \mathbb{R}^p$ (para desigualdades).

Classificação Numérica Lógica (Tolerância $\epsilon = 10^{-8}$):

Ponto Crítico KKT Válido (Convergência Global): Ocorre se, e somente se, o Erro KKT Consolidado ($E_{\text{KKT}}$) computado pela norma do supremo for menor ou igual à tolerância estrita $\epsilon$, e todos os multiplicadores de desigualdade forem não-negativos ($\mu_j \geq -\epsilon$).

Ponto Viável mas Não-Ótimo: Ocorre quando as restrições primais são atendidas ($\max(\|h(x)\|_\infty, \|\max(0, g(x))\|_\infty) \leq \epsilon$), porém há desbalanço no espaço dual (erro de estacionaridade elevado ou complementaridade violada).

Ponto Inviável: Ocorre se o erro associado às restrições primais excede a tolerância $\epsilon$, significando que o ponto viola os limites físicos do espaço de busca.

In [ ]:
import numpy as np

def calcular_erro_kkt_consolidado(grad_f, J_h=None, h_val=None, J_g=None, g_val=None, lambda_val=None, mu_val=None):
    """
    Função motor de resíduos desenvolvida na Atividade 3 (reutilizada como dependência).
    """
    grad_f = np.array(grad_f, dtype=float).flatten()
    h = np.array(h_val, dtype=float).flatten() if h_val is not None else np.array([])
    g = np.array(g_val, dtype=float).flatten() if g_val is not None else np.array([])
    lam = np.array(lambda_val, dtype=float).flatten() if lambda_val is not None else np.array([])
    mu = np.array(mu_val, dtype=float).flatten() if mu_val is not None else np.array([])

    R_est = np.copy(grad_f)
    if h.size > 0 and J_h is not None:
        R_est += np.dot(np.array(J_h, dtype=float).T, lam)
    if g.size > 0 and J_g is not None:
        R_est += np.dot(np.array(J_g, dtype=float).T, mu)

    R_eq = np.copy(h)
    R_ineq = np.maximum(0.0, g) if g.size > 0 else np.array([])
    R_comp = mu * g if g.size > 0 else np.array([])

    norm_est = np.max(np.abs(R_est)) if R_est.size > 0 else 0.0
    norm_eq = np.max(np.abs(R_eq)) if R_eq.size > 0 else 0.0
    norm_ineq = np.max(np.abs(R_ineq)) if R_ineq.size > 0 else 0.0
    norm_comp = np.max(np.abs(R_comp)) if R_comp.size > 0 else 0.0

    erro_kkt_max = max(norm_est, norm_eq, norm_ineq, norm_comp)

    return {
        "Erro_KKT_Consolidado": erro_kkt_max,
        "Residuo_Estacionaridade": norm_est,
        "Residuo_Igualdade": norm_eq,
        "Residuo_Desigualdade_Violada": norm_ineq,
        "Residuo_Complementaridade": norm_comp
    }


def arquitetura_validador_kkt(grad_f, J_h=None, h_val=None, J_g=None, g_val=None, lambda_val=None, mu_val=None, epsilon=1e-8):
    """
    Implementação Computacional da Atividade 4: Validador Arquitetural de Pontos Críticos.
    Classifica de forma inteligente e robusta o estado corrente do algoritmo de otimização.
    """
    # 1. Obter resíduos detalhados através do motor KKT
    residuos = calcular_erro_kkt_consolidado(grad_f, J_h, h_val, J_g, g_val, lambda_val, mu_val)

    mu = np.array(mu_val, dtype=float).flatten() if mu_val is not None else np.array([])

    # 2. Avaliar condições individuais de viabilidade primal e dual
    erro_primal = max(residuos["Residuo_Igualdade"], residuos["Residuo_Desigualdade_Violada"])
    viabilidade_primal_satisfeita = erro_primal <= epsilon

    # Testar se algum multiplicador mu viola a condição de não-negatividade (viabilidade dual)
    viabilidade_dual_satisfeita = True
    if mu.size > 0:
        if np.any(mu < -epsilon):
            viabilidade_dual_satisfeita = False

    erro_global = residuos["Erro_KKT_Consolidado"]

    # 3. Lógica de tomada de decisão/classificação do estado do Solver
    if erro_global <= epsilon and viabilidade_dual_satisfeita:
        classificacao = "PONTO CRÍTICO KKT VÁLIDO (Convergência Atingida com Sucesso)"
        codigo_status = 0
    elif viabilidade_primal_satisfeita:
        classificacao = "PONTO VIÁVEL MAS NÃO-ÓTIMO (Falta convergência nas condições de Estacionaridade/Dualidade)"
        codigo_status = 1
    else:
        classificacao = "PONTO INVIÁVEL (Viola as restrições físicas do problema)"
        codigo_status = -1

    # Agrupar dicionário de saída do relatório de validação
    relatorio = {
        "status_code": codigo_status,
        "classificacao": classificacao,
        "erro_kkt_max": erro_global,
        "detalhes_residuos": residuos,
        "viabilidade_primal_ok": viabilidade_primal_satisfeita,
        "viabilidade_dual_ok": viabilidade_dual_satisfeita
    }

    return relatorio


# --- Teste de Sanidade e Validação da Arquitetura ---
if __name__ == "__main__":
    print("=== CASO TESTE 1: Simulando um Ponto Ótimo Legítimo ===")
    gf_ok = [0.0, 0.0]
    relat_1 = arquitetura_validador_kkt(grad_f=gf_ok)
    print(f"Resultado: {relat_1['classificacao']}")
    print(f"Erro Máximo: {relat_1['erro_kkt_max']:.2e}\n")

    print("=== CASO TESTE 2: Simulando um Ponto Inviável (Restrição Estourada) ===")
    gf_inv = [0.0, 0.0]
    g_violado = [0.55] # g(x) = 0.55 > 0 viola g(x) <= 0
    relat_2 = arquitetura_validador_kkt(grad_f=gf_inv, g_val=g_violado)
    print(f"Resultado: {relat_2['classificacao']}")
    print(f"Resíduo de Desigualdade Violada: {relat_2['detalhes_residuos']['Residuo_Desigualdade_Violada']:.4f}")

=== CASO TESTE 1: Simulando um Ponto Ótimo Legítimo ===
Resultado: PONTO CRÍTICO KKT VÁLIDO (Convergência Atingida com Sucesso)
Erro Máximo: 0.00e+00

=== CASO TESTE 2: Simulando um Ponto Inviável (Restrição Estourada) ===
Resultado: PONTO INVIÁVEL (Viola as restrições físicas do problema)
Resíduo de Desigualdade Violada: 0.5500


Atividade 5 — Item a)

Enunciado:

a) Receber os pontos $(x_k, \mu_k)$. Inicialize na memória do computador os dados coletados do algoritmo após a execução de algumas iterações no problema do Veleiro: o ponto primal corrente $x_k = [2.5 \quad 2.5]^T$ e o multiplicador de Lagrange associado à restrição de desigualdade da reta $\mu_k = 5.0$.

Resposta Analítica e Teórica:

Os parâmetros numéricos fornecidos delimitam as coordenadas de estado do sistema dentro do espaço dual $\mathbb{R}^n \times \mathbb{R}^p$, com $n=2$ variáveis primais e $p=1$ restrição de desigualdade. Em computação científica voltada para a Ciência de Dados, o vetor contínuo $x_k$ representa a localização geográfica (posição cartesiana) do veleiro no plano. O escalar $\mu_k$ atua como a variável dual (força ou multiplicador), quantificando a sensibilidade ou magnitude da barreira física imposta sobre a minimização da função objetivo.

In [ ]:
import numpy as np

def atividade_5_item_a():
    """
    Inicializa na memória do computador os pontos de teste primal e dual
    fornecidos pelo enunciado da Atividade 5(a).
    """
    # Ponto primal corrente x_k (n = 2)
    x_k = np.array([2.5, 2.5], dtype=float)

    # Multiplicador de Lagrange mu_k (p = 1)
    mu_k = np.array([5.0], dtype=float)

    return x_k, mu_k

# Execução isolada do Item A
if __name__ == "__main__":
    x, mu = atividade_5_item_a()
    print("=== ATIVIDADE 5 — ITEM A ===")
    print(f"Ponto Primal x_k alocado: {x}")
    print(f"Multiplicador Dual mu_k alocado: {mu}")

=== ATIVIDADE 5 — ITEM A ===
Ponto Primal x_k alocado: [2.5 2.5]
Multiplicador Dual mu_k alocado: [5.]


Atividade 5 — Item b)

Enunciado:

b) Avaliar os resíduos lógicos. Mapeie analiticamente a função objetivo do Veleiro $f(x) = x_1^2 + x_2^2$ e a restrição de desigualdade da linha de areia $g(x) = 5 - x_1 - x_2 \leq 0$. A partir destas definições, calcule os valores numéricos dos resíduos das quatro dimensões KKT (Estacionaridade, Viabilidade Primal, Viabilidade Dual e Complementaridade) no ponto inicializado.

Resposta Analítica e Teórica:A modelagem analítica e as derivadas de primeira ordem aplicadas às equações explícitas do Veleiro fornecem as seguintes expressões matemáticas:


Gradiente da Objetivo: $\nabla f(x) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \end{bmatrix} = \begin{bmatrix} 2x_1 \\ 2x_2 \end{bmatrix} \implies \nabla f(2.5, 2.5) = \begin{bmatrix} 5.0 \\ 5.0 \end{bmatrix}$


Função de Restrição: $g(x) = 5 - x_1 - x_2 \implies g(2.5, 2.5) = 5 - 2.5 - 2.5 = 0.0$


Jacobiana da Restrição: $J_g(x) = \begin{bmatrix} \frac{\partial g}{\partial x_1} & \frac{\partial g}{\partial x_2} \end{bmatrix} = \begin{bmatrix} -1.0 & -1.0 \end{bmatrix}$

Fazendo a conexão direta com a Atividade 3, os resíduos matemáticos individuais são calculados por:Estacionaridade ($R_{\text{est}}$): $\nabla f(x_k) + J_g(x_k)^T \mu_k = \begin{bmatrix} 5.0 \\ 5.0 \end{bmatrix} + \begin{bmatrix} -1.0 \\ -1.0 \end{bmatrix} (5.0) = \begin{bmatrix} 5.0 - 5.0 \\ 5.0 - 5.0 \end{bmatrix} = \begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix}$

Viabilidade Primal ($R_{\text{prim}}$): Baseando-se no operador robusto, $R_{\text{prim}} = \max(0, g(x_k)) = \max(0, 0.0) = 0.0$ (o ponto é viável e a restrição está ativa na fronteira).

Viabilidade Dual ($R_{\text{dual}}$): $\max(0, -\mu_k) = \max(0, -5.0) = 0.0$ (sinal correto).Complementaridade ($R_{\text{comp}}$): $\mu_k \cdot g(x_k) = 5.0 \cdot 0.0 = 0.0$ (relação obedecida).

In [ ]:
import numpy as np

def atividade_5_item_b(x_k, mu_k):
    """
    Mapeia as funções analíticas do problema e avalia numericamente
    os resíduos lógicos das quatro dimensões KKT.
    """
    x1, x2 = x_k[0], x_k[1]

    # 1. Modelagem matemática analítica das estruturas
    grad_f = np.array([2.0 * x1, 2.0 * x2], dtype=float)
    g_val = np.array([5.0 - x1 - x2], dtype=float)
    J_g = np.array([[-1.0, -1.0]], dtype=float)

    # 2. Cálculo dos resíduos conforme sistema deduzido
    R_est = grad_f + np.dot(J_g.T, mu_k)
    R_prim = np.maximum(0.0, g_val)
    R_dual = np.maximum(0.0, -mu_k)
    R_comp = mu_k * g_val

    return R_est, R_prim, R_dual, R_comp

# Execução isolada do Item B
if __name__ == "__main__":
    x_teste, mu_teste = np.array([2.5, 2.5]), np.array([5.0])
    r_est, r_prim, r_dual, r_comp = atividade_5_item_b(x_teste, mu_teste)
    print("=== ATIVIDADE 5 — ITEM B ===")
    print(f"Resíduo de Estacionaridade (Vetor): {r_est}")
    print(f"Resíduo de Viabilidade Primal:      {r_prim}")
    print(f"Resíduo de Viabilidade Dual:        {r_dual}")
    print(f"Resíduo de Complementaridade:       {r_comp}")

=== ATIVIDADE 5 — ITEM B ===
Resíduo de Estacionaridade (Vetor): [0. 0.]
Resíduo de Viabilidade Primal:      [0.]
Resíduo de Viabilidade Dual:        [0.]
Resíduo de Complementaridade:       [0.]


Atividade 5 — Item c)

Enunciado:

c) Mostrar as saídas detalhadas do diagnóstico na tela. Desenvolva a rotina de formatação para exibir o diagnóstico do resolvedor na tela. A função deve unificar os resíduos do item anterior, extrair as suas respectivas normas infinitas $(\|\cdot\|_\infty)$ e computar o Erro KKT Máximo Consolidado, imprimindo um relatório técnico no console.

Resposta Analítica e Teórica:Para condensar as falhas multidimensionais calculadas pelo processador em um indicador escalar de parada estável, aplica-se a norma do supremo (maior valor absoluto encontrado nos blocos de vetores). O indicador unificado, $E_{\text{KKT}} = \max(\|R_{\text{est}}\|_\infty, \|R_{\text{prim}}\|_\infty, \|R_{\text{dual}}\|_\infty, \|R_{\text{comp}}\|_\infty)$, remove discrepâncias de dimensionalidade física e valida numericamente o ponto sob a tolerância global do resolvedor ($\epsilon = 10^{-8}$).

In [ ]:
import numpy as np

def atividade_5_item_c(r_est, r_prim, r_dual, r_comp):
    """
    Unifica as quatro dimensões de resíduos, extrai as normas infinitas
    e exibe o relatório de diagnóstico estruturado na tela.
    """
    # Extração das normas do supremo (Normas Infinitas)
    norm_est = np.max(np.abs(r_est))
    norm_prim = np.max(np.abs(r_prim))
    norm_dual = np.max(np.abs(r_dual))
    norm_comp = np.max(np.abs(r_comp))

    # Erro Máximo Consolidado KKT
    erro_kkt_max = max(norm_est, norm_prim, norm_dual, norm_comp)

    # Impressão rica de diagnóstico conforme padrão científico
    print("======= DIAGNÓSTICO DETALHADO DO SOLVER KKT =======")
    print(f"ERRO MÁXIMO CONSOLIDADO (E_KKT): {erro_kkt_max:.6e}")
    print("-" * 51)
    print(f"-> Norma de Estacionaridade:     {norm_est:.6e}")
    print(f"-> Norma de Viabilidade Primal:  {norm_prim:.6e}")
    print(f"-> Norma de Viabilidade Dual:    {norm_dual:.6e}")
    print(f"-> Norma de Complementaridade:   {norm_comp:.6e}")
    print("===================================================")

    return erro_kkt_max

# Execução isolada do Item C
if __name__ == "__main__":
    # Testando com os resíduos nulos obtidos matematicamente no item b
    r_e, r_p, r_d, r_c = np.array([0.0, 0.0]), np.array([0.0]), np.array([0.0]), np.array([0.0])
    atividade_5_item_c(r_e, r_p, r_d, r_c)

======= DIAGNÓSTICO DETALHADO DO SOLVER KKT =======
ERRO MÁXIMO CONSOLIDADO (E_KKT): 0.000000e+00
---------------------------------------------------
-> Norma de Estacionaridade:     0.000000e+00
-> Norma de Viabilidade Primal:  0.000000e+00
-> Norma de Viabilidade Dual:    0.000000e+00
-> Norma de Complementaridade:   0.000000e+00


 Atividade 5 — Item d)

 Enunciado:

 d) Interpretação Física (O Barco Continua Encalhado?). Com base no Erro KKT Máximo Consolidado e no comportamento das variáveis primais e duais, forneça a interpretação física e geométrica definitiva do problema: O barco continua encalhado ou o motor pode ser ligado?

 Resposta Analítica e Teórica:Como o Erro KKT Máximo Consolidado extraído foi exatamente 0.000000e+00, conclui-se sob a ótica teórica de Martínez e Santos que o ponto de teste $x_k = [2.5 \quad 2.5]^T$ é rigorosamente um Ponto Crítico KKT Válido (mínimo local ótimo do problema restrito).

 Mapeando a física do cenário:

 O resíduo $g(x_k) = 0.0$ indica que a restrição de desigualdade está ativa. Geometricamente, o veleiro encontra-se posicionado exatamente em cima da curva de fronteira (a linha divisória do banco de areia).

 O multiplicador de Lagrange associado é estritamente positivo ($\mu_k = 5.0 > 0$). Isso comprova que a restrição está gerando uma força geométrica ativa e não-nula que se opõe perfeitamente ao gradiente de descida livre da função objetivo ($\nabla f(x_k) = [5 \quad 5]^T$).

 Como as forças estão anuladas na fronteira e o multiplicador dual é positivo, qualquer tentativa de mover o barco em direção à origem (mínimo irrestrito $[0 \quad 0]^T$) causaria o transbordo da restrição ($g(x) > 0$). Portanto, o barco continua fisicamente encalhado na areia e o motor não deve ser ligado para forçar essa direção, sob pena de violação da viabilidade do sistema.

In [ ]:
import numpy as np

def atividade_5_item_d(erro_kkt, g_val, mu_val, epsilon=1e-8):
    """
    Atividade 5 — Item d).
    Avalia as variáveis primais e duais para emitir a interpretação
    física e geométrica definitiva sobre o status do veleiro.
    """
    print("\n=== INTERPRETAÇÃO GEOMÉTRICA E FÍSICA DO VELEIRO ===")

    # Forçar a conversão segura para escalares flutuantes extraindo o primeiro elemento
    g_escalar = float(np.array(g_val).flatten()[0])
    mu_escalar = float(np.array(mu_val).flatten()[0])
    erro_escalar = float(erro_kkt)

    if erro_escalar <= epsilon:
        # Critério de restrição ativa (|g| próximo de 0) com força dual aplicada (mu > 0)
        if np.abs(g_escalar) <= epsilon and mu_escalar > epsilon:
            print("DIAGNÓSTICO CONCLUSIVO: O barco CONTINUA ENCALHADO!")
            print(f"Justificativa: A restrição está ativa (g = {g_escalar}) e o multiplicador")
            print(f"dual positivo (mu = {mu_escalar}) prova que a fronteira de areia anula")
            print("o gradiente descendente. Ligar o motor violaria a viabilidade primal.")
        else:
            print("DIAGNÓSTICO CONCLUSIVO: O barco está livre! O motor pode ser ligado.")
    else:
        print("DIAGNÓSTICO CONCLUSIVO: O sistema não convergiu para um ponto estacionário.")
    print("====================================================\n")

# Execução isolada de teste (Aceita listas, arrays ou números puros sem quebrar)
if __name__ == "__main__":
    # Dados vindos do Item C e Item B
    erro_do_item_c = 0.0
    g_do_item_b = 0.0
    mu_do_item_a = 5.0

    atividade_5_item_d(erro_kkt=erro_do_item_c, g_val=g_do_item_b, mu_val=mu_do_item_a)


=== INTERPRETAÇÃO GEOMÉTRICA E FÍSICA DO VELEIRO ===
DIAGNÓSTICO CONCLUSIVO: O barco CONTINUA ENCALHADO!
Justificativa: A restrição está ativa (g = 0.0) e o multiplicador
dual positivo (mu = 5.0) prova que a fronteira de areia anula
o gradiente descendente. Ligar o motor violaria a viabilidade primal.



Atividade 6 — Item a) Dedução do Sistema de Newton em Direções Tangentes

Enunciado Teórico:

Para um problema com restrições de igualdade da forma $\min f(x)$ sujeito a $h(x) = 0$, as condições KKT de primeira ordem exigem que o gradiente do Lagrangiano e os resíduos das restrições se anulem: $\nabla \mathcal{L}(x, \lambda) = 0$ e $h(x) = 0$. Desenvolva o embasamento analítico que justifica a linearização dessas equações via expansão em série de Taylor de primeira ordem, demonstrando de onde surge o sistema linear em blocos iterativo.

Resposta Analítica e Teórica:Considerando o sistema não-linear de equações dado por:

$$F(x, \lambda) = \begin{bmatrix} \nabla f(x) + J_h(x)^T \lambda \\ h(x) \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$

Aplicando o Método de Newton multidimensional a partir de uma estimativa corrente $(x_k, \lambda_k)$, realiza-se uma aproximação linear (Série de Taylor) na vizinhança do ponto:$$F(x_k, \lambda_k) + J_F(x_k, \lambda_k) \begin{bmatrix} dx \\ d\lambda \end{bmatrix} \approx \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$Onde $J_F$ é a matriz Jacobiana do sistema de leis KKT (a qual equivale à matriz Hessiana em blocos). Calculando as derivadas parciais do sistema $F$, obtemos a matriz do sistema:

$$J_F(x_k, \lambda_k) = \begin{bmatrix} \nabla_{xx}^2 \mathcal{L}(x_k, \lambda_k) & J_h(x_k)^T \\ J_h(x_k) & 0 \end{bmatrix}$$

Substituindo $J_F$ e isolando o termo incremental $\begin{bmatrix} dx \\ d\lambda \end{bmatrix}$, deduz-se o sistema linear em blocos oficial que o processador deve computar a cada iteração:

$$\begin{bmatrix} \nabla_{xx}^2 \mathcal{L}(x_k, \lambda_k) & J_h(x_k)^T \\ J_h(x_k) & 0 \end{bmatrix} \begin{bmatrix} dx \\ d\lambda \end{bmatrix} = \begin{bmatrix} -\nabla f(x_k) - J_h(x_k)^T \lambda_k \\ -h(x_k) \end{bmatrix}$$

In [ ]:
import numpy as np

def deduzir_blocos_rhs(grad_f, J_h, h_val, lambda_k):
    """
    Atividade 6 - Item a).
    Calcula analiticamente o lado direito (RHS) do sistema de Newton-KKT,
    que representa o gradiente negativo do Lagrangiano e o residuo de h.
    """
    g_f = np.array(grad_f, dtype=float).flatten()
    A = np.array(J_h, dtype=float)
    h = np.array(h_val, dtype=float).flatten()
    lam = np.array(lambda_k, dtype=float).flatten()

    # Gradiente do Lagrangiano: grad_L = grad_f + J_h^T * lambda
    grad_L = g_f + np.dot(A.T, lam)

    # Construcao do vetor RHS: [-grad_L, -h]
    vetor_rhs = np.concatenate([-grad_L, -h])
    return vetor_rhs

if __name__ == "__main__":
    print("=== ATIVIDADE 6 - ITEM A ===")
    rhs = deduzir_blocos_rhs(grad_f=[1.0, 2.0], J_h=[[1.0, 1.0]], h_val=[0.1], lambda_k=[0.5])
    print(f"Vetor Lado Direito (RHS) linearizado: {rhs}")

=== ATIVIDADE 6 - ITEM A ===
Vetor Lado Direito (RHS) linearizado: [-1.5 -2.5 -0.1]


Atividade 6 — Item b) Implementação do Loop de Otimização e Critério de Parada

Enunciado Teórico:

Utilizando as funções de montagem de matriz em blocos da Atividade 5 e o validador lógico estruturado na Atividade 4, codifique a estrutura principal do laço iterativo (while). O algoritmo deve interromper a execução caso o erro KKT consolidado seja menor ou igual à tolerância alvo $\epsilon = 10^{-8}$ ou caso atinja o limite de segurança de $100$ iterações.

Resposta Analítica e Teórica:

O controle do fluxo de execução em resolvedores contínuos mapeia o comportamento assintótico dos resíduos. A cada passo, o vetor de estados é atualizado por $x_{k+1} = x_k + dx$ e $\lambda_{k+1} = \lambda_k + d\lambda$. O laço iterativo garante que a memória consuma ciclos de CPU apenas enquanto as restrições e a estacionaridade violarem a vizinhança do zero numérico mapeada pela norma infinita, servindo como uma barreira robusta contra divergências e loops infinitos.

In [ ]:
import numpy as np

def run_loop_newton_kkt(funcao_problema, x_inicial, lambda_inicial, epsilon=1e-8, max_iter=100):
    """
    Atividade 6 - Item b).
    Estrutura o laco iterativo principal do solver utilizando o criterio de parada KKT.
    """
    x_k = np.array(x_inicial, dtype=float).flatten()
    lam_k = np.array(lambda_inicial, dtype=float).flatten()

    hist_erro = []
    print("=== INICIALIZANDO LOOP ITERATIVO (ITEM B) ===")

    k = 0
    while k < max_iter:
        # 1. Coletar matrizes analiticas do problema no ponto corrente
        grad_f, Hess_L, J_h, h_val = funcao_problema(x_k, lam_k)

        # 2. Calcular o erro corrente usando a logica unificada
        grad_L = grad_f + np.dot(J_h.T, lam_k)
        erro_kkt = max(np.max(np.abs(grad_L)), np.max(np.abs(h_val)))
        hist_erro.append(erro_kkt)

        print(f"Iteracao {k:02d} | Erro KKT Consolidado: {erro_kkt:.6e}")

        # Criterio de Parada Oficial
        if erro_kkt <= epsilon:
            print(f"-> CONVERGÊNCIA ATINGIDA NA ITERAÇÃO {k}!\n")
            break

        # 3. Montar e resolver o sistema em blocos
        bloco_zeros = np.zeros((J_h.shape[0], J_h.shape[0]))
        matriz_kkt = np.block([[Hess_L, J_h.T], [J_h, bloco_zeros]])
        vetor_rhs = np.concatenate([-grad_L, -h_val])

        # Passo de Newton
        passo = np.linalg.solve(matriz_kkt, vetor_rhs)

        # 4. Atualizar estados em memoria
        dx = passo[:len(x_k)]
        d_lam = passo[len(x_k):]

        x_k += dx
        lam_k += d_lam
        k += 1

    return x_k, lam_k, hist_erro

Atividade 6 — Item c) Validação Numérica Num Problema Quadrático Restrito

Enunciado Teórico:

Aplique o algoritmo completo desenvolvido no item anterior para resolver o problema de minimização quadrática com restrição linear de igualdade definido por:

$$\min f(x) = x_1^2 + x_2^2 \quad \text{sujeito a:} \quad h(1) = x_1 + x_2 - 2 = 0$$

Adote o ponto inicial de busca $x_0 = [10.0, \ 5.0]^T$ e $\lambda_0 = [0.0]^T$. Demonstre em quantas iterações o Método de Newton converge e comente sobre a velocidade de convergência em problemas quadráticos.

Resposta Analítica e Teórica:

Para o problema proposto, as estruturas analíticas de derivadas são:

Gradiente e Hessiana de $f(x)$: $\nabla f(x) = \begin{bmatrix} 2x_1 \\ 2x_2 \end{bmatrix}$, $\nabla^2 f(x) = \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix}$

Jacobiana e Resíduo de $h(x)$: $J_h(x) = \begin{bmatrix} 1 & 1 \end{bmatrix}$, $h(x) = x_1 + x_2 - 2$

Hessiana do Lagrangiano: Como $h(x)$ é linear, sua segunda derivada é nula, logo $\nabla_{xx}^2 \mathcal{L} = \nabla^2 f(x) = \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix}$ (matriz constante).

O Método de Newton estende a propriedade fundamental de convergência em um único passo para sistemas lineares. Como o Lagrangiano de um problema de programação quadrática com restrições lineares gera um sistema de equações estritamente linear, o algoritmo deve atingir a convergência exata com erro próximo a zero em exatamente 1 iteração, independentemente da distância do ponto inicial $x_0$.

In [ ]:
import numpy as np

def simular_problema_quadratico(x_vetor, lam_vetor):
    """
    Modela numericamente as equações analíticas do problema quadrático de teste.
    """
    x1, x2 = x_vetor[0], x_vetor[1]

    grad_f = np.array([2.0 * x1, 2.0 * x2])
    Hess_L = np.array([[2.0, 0.0],
                       [0.0, 2.0]]) # Matriz quadrada constante
    J_h = np.array([[1.0, 1.0]])
    h_val = np.array([x1 + x2 - 2.0])

    return grad_f, Hess_L, J_h, h_val

# Pipeline Unificado de Execução do Teste no Colab
if __name__ == "__main__":
    # Parametros fornecidos pelo enunciado
    x_init = np.array([10.0, 5.0])
    lam_init = np.array([0.0])

    # Invocar o solver estruturado no item b
    x_opt, lam_opt, historico = run_loop_newton_kkt(simular_problema_quadratico, x_init, lam_init)

    print("======= RESULTADOS DO DIAGNÓSTICO FINAL =======")
    print(f"Ponto Ótimo Primal Encontrado x*: {x_opt} (Esperado analiticamente: [1. 1.])")
    print(f"Multiplicador Dual Encontrado lambda*: {lam_opt}")
    print(f"Total de Iteracoes executadas: {len(historico) - 1}")
    print("===============================================")

=== INICIALIZANDO LOOP ITERATIVO (ITEM B) ===
Iteracao 00 | Erro KKT Consolidado: 2.000000e+01
Iteracao 01 | Erro KKT Consolidado: 0.000000e+00
-> CONVERGÊNCIA ATINGIDA NA ITERAÇÃO 1!

======= RESULTADOS DO DIAGNÓSTICO FINAL =======
Ponto Ótimo Primal Encontrado x*: [1. 1.] (Esperado analiticamente: [1. 1.])
Multiplicador Dual Encontrado lambda*: [-2.]
Total de Iteracoes executadas: 1


Atividade 6 — Item d) Diagnóstico de Desempenho e Velocidade de Convergência

Enunciado Teórico:

Apresente uma análise técnica baseada nos resultados numéricos obtidos no item anterior. Explique a taxa de convergência observada e discuta o comportamento do resíduo de estacionaridade face às propriedades matemáticas do sistema linearizado.

Resposta Analítica e Teórica:

O experimento numérico confirma a previsão teórica: partindo de um ponto significativamente distante da solução viável ($x_0 = [10, 5]^T$, cujo erro inicial era da ordem de $1.5 \times 10^1$), o resolvedor despencou o erro KKT diretamente para 0.000000e+00 na iteração 1.

Esse comportamento representa o fenômeno da Convergência Quadrática no Limite, intrínseco aos métodos baseados em aproximações de Newton. Em Ciência de Dados e Otimização Computacional, quando a função objetivo é uma parábola perfeita (quadrática) e as superfícies de restrição são hiperpalanos (lineares), o modelo local gerado pela Série de Taylor deixa de ser uma aproximação e passa a ser a representação exata e global da geometria do problema. Desse modo, o processador calcula as direções tangentes exatas no primeiro passo, zerando o vetor de resíduos de estacionaridade de forma instantânea.